# COSMOS

The goal is to find the tract and patch of a DDF field.
As such , it is possible to make a Data Query SQL request based on tract and patch.
We use DP2 to get the Rubin Skymapping.

- author : Sylvie Dagoret-Campagne
- creation-date : 2026-10-25

In [ ]:
import warnings

warnings.filterwarnings(
    "error",
    category=FutureWarning,
    message=".*Butler.collections.*",
)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

import numpy as np
import lsst.geom as geom

## Configuration of Butler

In [ ]:
from lsst.daf.butler import Butler

butler_url = "dp2"
repo = "dp2"

collection = "dp2"
instrument = "LSSTCam"
skymapName = "lsst_cells_v2"
butler = Butler(repo)

In [ ]:
# Derived configuration (do not edit)
where_clause = f"instrument = '{instrument}'"


# Initialise the Butler repository
butler = Butler(repo, collections=collection)
registry = butler.registry
print(f"Butler initialised | repo: {repo}")
print(f"Collection : {collection}")

In [ ]:
# Rubin sky tessellation (tracts, patches, cells) used by DP2
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)

## Define DDF coordinates

In [ ]:
lsstcam_fields = {}

# =========================
# Deep Drilling Fields (LSSTCam)
# RA, Dec of the field centres in degrees (ICRS)
# =========================
lsstcam_fields["COSMOS"] = {"field_name": "COSMOS Deep Drilling Field", "ra": 150.11, "dec": 2.23}
lsstcam_fields["XMM-LSS"] = {"field_name": "XMM-LSS Deep Drilling Field", "ra": 35.57, "dec": -4.82}
lsstcam_fields["ELAIS-S1"] = {"field_name": "ELAIS-S1 Deep Drilling Field", "ra": 9.45, "dec": -44.02}
lsstcam_fields["ECDFS"] = {"field_name": "Extended Chandra Deep Field South", "ra": 52.98, "dec": -28.12}
# Euclid Deep Field South = 2 pointings LSSTCam
lsstcam_fields["EDFS_a"] = {"field_name": "Euclid Deep Field South (pointing a)", "ra": 58.90, "dec": -49.32}
lsstcam_fields["EDFS_b"] = {"field_name": "Euclid Deep Field South (pointing b)", "ra": 63.60, "dec": -47.60}


In [ ]:
lsstcam_fields["COSMOS"]["ra"]

## Find collections

In [ ]:
collections = list(butler.collections.query("*"))

In [ ]:
# List the DP2 collections. The loop variable is `coll` so that the
# `collection` defined in the configuration cell is not overwritten.
for coll in collections:
    if "DP2" in coll:
        print(coll)

print(f"\nNumber of collections: {len(collections)}")

## Find Tract and Patch

In [ ]:
# Search region: a square of side `field_size` centred on the COSMOS DDF
ra_center = lsstcam_fields["COSMOS"]["ra"]
dec_center = lsstcam_fields["COSMOS"]["dec"]
field_size = 3.0  # side of the square search region [deg]
half_size = field_size / 2.0

# Regular grid of sky positions covering the search region
ras = np.linspace(ra_center - half_size, ra_center + half_size, 21)
decs = np.linspace(dec_center - half_size, dec_center + half_size, 21)

coord_list = [
    geom.SpherePoint(ra * geom.degrees, dec * geom.degrees)
    for ra in ras
    for dec in decs
]

# All (tract, patches) pairs overlapping the region
tract_patch_list = skymap.findTractPatchList(coord_list)

# {tract_id: set of patch sequential indices}, reused in the plots below
selected_patches = {}

for tract_info, patch_infos in tract_patch_list:
    patch_ids = sorted({
        patch.getSequentialIndex()
        for patch in patch_infos
    })
    selected_patches[tract_info.getId()] = set(patch_ids)

    print(
        f"Tract {tract_info.getId()}: "
        f"{len(patch_ids)} patches {patch_ids}"
    )

## Plot tracts and patches

In [ ]:
def patch_boundary_sky(tract_info, bbox, n=30):
    """Outline of a pixel bounding box of a tract, converted to RA/Dec.

    Works for a patch (inner or outer bbox) as well as for a whole tract bbox.
    Each side is sampled with `n` points and converted with the tract WCS,
    so that the curvature of the projection is rendered correctly.
    Returns (ra, dec) in degrees, as a closed polygon.
    """
    wcs = tract_info.getWcs()

    xmin, ymin = bbox.getMinX(), bbox.getMinY()
    xmax, ymax = bbox.getMaxX(), bbox.getMaxY()

    # Sample the four sides of the rectangle
    x = np.linspace(xmin, xmax, n)
    y = np.linspace(ymin, ymax, n)

    pixels = (
        [(xx, ymin) for xx in x]
        + [(xmax, yy) for yy in y[1:]]
        + [(xx, ymax) for xx in x[-2::-1]]
        + [(xmin, yy) for yy in y[-2:0:-1]]
    )

    sky = [
        wcs.pixelToSky(geom.Point2D(float(xx), float(yy)))
        for xx, yy in pixels
    ]

    ra = np.array([p.getRa().asDegrees() for p in sky])
    dec = np.array([p.getDec().asDegrees() for p in sky])

    # Repeat the first point so that the outline is a closed polygon
    # (the sampling above stops one point short of the starting corner)
    return np.append(ra, ra[0]), np.append(dec, dec[0])


In [ ]:
# Centre of the COSMOS DDF (the displayed window is ra0 +/- half_size, dec0 +/- half_size)
ra0, dec0 = ra_center, dec_center

# Tracts overlapping the search region, taken directly from findTractPatchList
tract_ids = set(selected_patches)
print("Tracts found:", sorted(tract_ids))

In [ ]:
import matplotlib.patheffects as pe
from matplotlib.colors import to_rgba
from matplotlib.patches import Patch

# One colour per tract, reused in every figure
tract_colors = {
    tract_id: plt.cm.tab10(i % 10)
    for i, tract_id in enumerate(sorted(tract_ids))
}


def inside_window(ra, dec):
    """True where (ra, dec) [deg] falls inside the displayed window."""
    return (np.abs(ra - ra0) <= half_size) & (np.abs(dec - dec0) <= half_size)


def visible_center_sky(tract_info, n=40):
    """Mean sky position of the part of a tract lying inside the window.

    Used to place the tract label where it is actually visible, even when the
    geometric centre of the tract is outside the plotted field.
    Returns None if the tract does not overlap the window.
    """
    bbox = tract_info.getBBox()
    wcs = tract_info.getWcs()
    xs = np.linspace(bbox.getMinX(), bbox.getMaxX(), n)
    ys = np.linspace(bbox.getMinY(), bbox.getMaxY(), n)
    sky = [
        wcs.pixelToSky(geom.Point2D(float(x), float(y)))
        for x in xs
        for y in ys
    ]
    ra = np.array([p.getRa().asDegrees() for p in sky])
    dec = np.array([p.getDec().asDegrees() for p in sky])
    mask = inside_window(ra, dec)
    if not mask.any():
        return None
    return ra[mask].mean(), dec[mask].mean()


def draw_ddf_centre(ax):
    """Mark the nominal centre of the DDF."""
    ax.plot(
        ra0, dec0, marker="*", markersize=16, color="gold",
        markeredgecolor="black", markeredgewidth=0.8, linestyle="none",
        zorder=6, label="COSMOS DDF centre",
    )


def style_sky_axes(ax, title):
    """Common formatting of the sky-map axes."""
    # RA increases towards the left (East is on the left of the sky)
    ax.set_xlim(ra0 + half_size, ra0 - half_size)
    ax.set_ylim(dec0 - half_size, dec0 + half_size)
    # Compensate the cos(dec) compression of RA so that angles are undistorted
    ax.set_aspect(1.0 / np.cos(np.deg2rad(dec0)))
    ax.ticklabel_format(useOffset=False)
    ax.set_xlabel("Right Ascension [deg]", fontsize=12)
    ax.set_ylabel("Declination [deg]", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.tick_params(labelsize=10, direction="in", top=True, right=True)
    ax.grid(True, alpha=0.25)


def add_legend(ax, extra_handles=()):
    """Legend with one entry per tract, plus optional extra entries."""
    tract_handles = [
        Patch(
            facecolor=to_rgba(tract_colors[t], 0.3),
            edgecolor=tract_colors[t],
            linewidth=2,
            label=f"Tract {t}",
        )
        for t in sorted(tract_ids)
    ]
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(
        handles=tract_handles + list(extra_handles) + handles,
        loc="upper right", fontsize=9, framealpha=0.9,
    )

In [ ]:
# Figure 1: tracts overlapping the COSMOS DDF
fig, ax = plt.subplots(figsize=(9, 8))

for tract_id in sorted(tract_ids):
    tract_info = skymap[tract_id]
    color = tract_colors[tract_id]

    # Outline of the full tract (edges are sampled so that the curvature of
    # the projection is rendered correctly)
    ra, dec = patch_boundary_sky(tract_info, tract_info.getBBox(), n=50)
    ax.fill(ra, dec, facecolor=to_rgba(color, 0.15), edgecolor="none", zorder=1)
    ax.plot(ra, dec, color=color, linewidth=2.5, zorder=2)

    # Tract ID, placed at the centre of the part of the tract that is visible
    center = visible_center_sky(tract_info)
    if center is not None:
        ax.text(
            *center, str(tract_id),
            fontsize=16, fontweight="bold", color=color,
            ha="center", va="center", zorder=4,
            path_effects=[pe.withStroke(linewidth=3, foreground="white")],
        )

draw_ddf_centre(ax)
style_sky_axes(ax, f"Rubin SkyMap ({skymapName}) — COSMOS DDF: tracts")
add_legend(ax)

plt.tight_layout()
# fig.savefig("cosmos_tracts.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 2: patches of the tracts overlapping the COSMOS DDF.
# Patches overlapping the search region are filled and numbered in bold;
# the other patches are drawn in the background for context.
fig, ax = plt.subplots(figsize=(12, 10))

for tract_id in sorted(tract_ids):
    tract_info = skymap[tract_id]
    wcs = tract_info.getWcs()
    color = tract_colors[tract_id]
    selected = selected_patches.get(tract_id, set())

    for patch_info in tract_info:
        patch_index = patch_info.getSequentialIndex()
        is_selected = patch_index in selected

        # InnerBBox: nominal patch outline, without overlap between patches
        bbox = patch_info.getInnerBBox()
        ra, dec = patch_boundary_sky(tract_info, bbox, n=8)

        # Skip patches that are entirely outside the displayed window
        if not inside_window(ra, dec).any():
            continue

        ax.fill(
            ra, dec,
            facecolor=to_rgba(color, 0.30) if is_selected else "none",
            edgecolor=to_rgba(color, 0.95 if is_selected else 0.40),
            linewidth=1.0 if is_selected else 0.6,
            zorder=2 if is_selected else 1,
        )

        # Patch index at the centre of its box (only if inside the window)
        cx = (bbox.getMinX() + bbox.getMaxX()) / 2
        cy = (bbox.getMinY() + bbox.getMaxY()) / 2
        center = wcs.pixelToSky(geom.Point2D(cx, cy))
        ra_c = center.getRa().asDegrees()
        dec_c = center.getDec().asDegrees()

        if inside_window(ra_c, dec_c):
            ax.text(
                ra_c, dec_c, str(patch_index),
                fontsize=7,
                fontweight="bold" if is_selected else "normal",
                color="black" if is_selected else "0.45",
                ha="center", va="center", zorder=3,
            )

    # Thick outline of the tract, drawn on top of its patches
    ra, dec = patch_boundary_sky(tract_info, tract_info.getBBox(), n=50)
    ax.plot(ra, dec, color=color, linewidth=2.5, zorder=4)

    # Large, faint tract ID in the background
    center = visible_center_sky(tract_info)
    if center is not None:
        ax.text(
            *center, str(tract_id),
            fontsize=40, fontweight="bold", color=color, alpha=0.25,
            ha="center", va="center", zorder=0,
        )

draw_ddf_centre(ax)
style_sky_axes(ax, f"Rubin SkyMap ({skymapName}) — COSMOS DDF: tracts and patches")
add_legend(ax, extra_handles=[
    Patch(facecolor=to_rgba("0.5", 0.3), edgecolor="0.3",
          label="Patch overlapping search region"),
    Patch(facecolor="none", edgecolor="0.6", label="Other patches"),
])

plt.tight_layout()
# fig.savefig("cosmos_tracts_patches.png", dpi=200, bbox_inches="tight")
plt.show()